<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/gem_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Dependencies

!pip install -q transformers torch
!pip install -q bitsandbytes

# first-party
from typing import Dict, Any, Optional, Tuple, List
import random
import time
import csv
import os

# third-party
from google.colab import drive
import pandas as pd
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import torch.nn.functional as F
import numpy as np


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00


In [2]:
# @title Paths and definitions

# --- folders ---

# mount GDrive
drive.mount('/content/drive')

# dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"
# create dataset directory (if not already existing)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# raw directory
RAW_DIR = "/content/drive/MyDrive/Thesis/raw/"
# create raw directory (if not already existing)
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# result directory
RESULT_DIR = "/content/drive/MyDrive/Thesis/raw/gem"
# create result directory (if not already existing)
os.makedirs(RESULT_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RESULT_DIR}.")

# --- call-settings---

# secret key
OPENAI_KEY = "Set key here"

# simulation/api-call
SIMULATE = True

# hardcoded client-values
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct" # as Xu et al.
TEMPERATURE_DEFAULT = 0.0
MAX_TOKENS = 4000
MAX_RETRIES = 3
RETRY_DELAY = 2.0
SYSTEM_PROMPT_GEM = """

You are the second reviewer for a scientific paper.
You are given the abstract of the paper and a list of review judgments from the first reviewer, starting with ’The reviewer appreciates/criticizes/questions/suggests’.
Your task is to provide your own judgments of the paper based on the given materials.
You should create a separate line for each judgment you have, starting with ’The reviewer appreciates/criticizes/questions/suggests’.
Ensure your judgments are concise, excluding specific details about the paper’s content.

""" # hardcoded as Xu et al.

# model settings
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True, # as Xu et al.
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# --- files and columns ---

# paper text
DATASET_PAPER = "dataset_paper.parquet"

# standardised human reviews
HUM_RES_1 = "std_human_revw_1_res.csv"
HUM_RES_2 = "std_human_revw_2_res.csv"
HUM_RES_3 = "std_human_revw_3_res.csv"

# assw
ASSW_RES = "assw_res.csv"

# standardised llm reviews
LLM_RES = "llm_sim_results.csv"

# col for merge
COMMON_COL = ['model', 'temperature', 'reasoning_effort', 'chain_of_thought']

# col for grouping
LLM_CONFIG_COLS = ['model_llm', 'temperature_llm', 'reasoning_effort_llm', 'chain_of_thought_llm']

# candidate columns for conditional review
HUMAN_REVIEW_COLS = ['human_review1', 'human_review2', 'human_review3']

# set review Y to the LLM review text column
TARGET_COL = 'review_text'

# define columns of result file
BASE_RES_COLUMNS = ['paper_id',
                  'forced_output',
                  'log_prob',
                  'perplexity',
                  'status',
                  'processed_timestamp',
                  'model', 'temperature', 'reasoning_effort', 'chain_of_thought']

# define columns of log file
BASE_LOG_COLUMNS = ['model',
                    'temperature',
                    'reasoning_effort',
                    'chain_of_thought',
                    'papers_processed_count',
                    'papers_succeeded_count',
                    'configuration_status']

# --- Predefined Calculation Settings ---

CALCULATION_SETTINGS = {
    # Conditional Probabilities (include_first_review_flag = True)
    "GEM_conditional": {
        "include_abstract_flag": False,
        "include_assw_flag": False,
        "include_first_review_flag": True
    },
    "GEM-S_conditional": {
        "include_abstract_flag": True,
        "include_assw_flag": False,
        "include_first_review_flag": True
    },
    "GEM-S_ASSW_conditional": {
        "include_abstract_flag": True,
        "include_assw_flag": True,
        "include_first_review_flag": True
    },
    # Marginal Probabilities (include_first_review_flag = False)
    "GEM_marginal": {
        "include_abstract_flag": False,
        "include_assw_flag": False,
        "include_first_review_flag": False
    },
    "GEM-S_marginal": {
        "include_abstract_flag": True,
        "include_assw_flag": False,
        "include_first_review_flag": False
    },
    "GEM-S_ASSW_marginal": {
        "include_abstract_flag": True,
        "include_assw_flag": True,
        "include_first_review_flag": False
    }
}

Mounted at /content/drive
Ensured dataset directory exists at: /content/drive/MyDrive/Thesis/dataset/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/raw/.


In [ ]:
# @title Load data

# define paths
file_paths = {
    'df_hum_1': RAW_DIR + HUM_RES_1,
    'df_hum_2': RAW_DIR + HUM_RES_2,
    'df_hum_3': RAW_DIR + HUM_RES_3,
    'df_llm': RAW_DIR + LLM_RES,
    'df_assw': RAW_DIR + ASSW_RES,
    'df_paper': DATASET_DIR + DATASET_PAPER
}

# initialize yet empty list of dataframes
dataframes = {}

# iterate over defined paths
for df_name, path in file_paths.items():
    current_df = None
    try:
        # try to load as csv
        try:
            current_df = pd.read_csv(path, engine='python')
            print(f"\nSuccessfully loaded {df_name} as CSV (python engine) from: {path}")
        except Exception as csv_load_e:
            print(f"Attempting to load {df_name} as Parquet after CSV failure (Error: {str(csv_load_e)}).")

            # try to load as parquet
            try:
                current_df = pd.read_parquet(path)
                print(f"Successfully loaded {df_name} as Parquet from: {path}")
            except Exception as parquet_load_e:

                raise Exception(f"Failed to load {df_name} from {path} as CSV or Parquet. CSV error: {str(csv_load_e)}, Parquet error: {str(parquet_load_e)}")

        if current_df is not None:

            # enforce correct format on columns on which merge is conditioned
            if df_name in ['df_hum_1', 'df_hum_2', 'df_hum_3', 'df_llm', 'df_assw'] and all(col in current_df.columns for col in COMMON_COL):
                current_df['model'] = current_df['model'].astype(str)
                current_df['temperature'] = current_df['temperature'].astype(float)
                current_df['reasoning_effort'] = current_df['reasoning_effort'].astype(str)
                current_df['chain_of_thought'] = current_df['chain_of_thought'].astype(str)
                print(f"Converted 'model', 'temperature', 'reasoning_effort', 'chain_of_thought' in {df_name} to specified types.")

            # check
            dataframes[df_name] = current_df
            print(f"\nDataFrame '{df_name}' Head:")
            display(dataframes[df_name].head())

            print(f"\nDataFrame '{df_name}' Columns:")
            print(dataframes[df_name].columns.tolist())

    except FileNotFoundError:
        print(f"ERROR: File not found at {path} for {df_name}. Please check the path and file type.")
    except Exception as e:
        print(f"An error occurred while reading the file for {df_name}: {str(e)}")


In [4]:
# @title Merge human reviews

# access dataframes
df_hum_1 = dataframes.get('df_hum_1')
df_hum_2 = dataframes.get('df_hum_2')
df_hum_3 = dataframes.get('df_hum_3')
df_assw = dataframes.get('df_assw')

# define common columns for this merge
common_cols = ['paper_id'] + COMMON_COL

# check if dataframes exist before merging
if df_hum_1 is None or df_hum_2 is None or df_hum_3 is None or df_assw is None:
    print("Error: One or more human review dataframes or the assw datafram are not loaded. Please ensure df_hum_1, df_hum_2, and df_hum_3 and df_assw are available.")
else:
    # merged_df = human review 1 + human review 2
    merged_df = pd.merge(df_hum_1, df_hum_2, on=common_cols, how='outer', suffixes=('', '_hum2'))

    # merged_df = merged_df + human review 3
    merged_df = pd.merge(merged_df, df_hum_3, on=common_cols, how='outer', suffixes=('', '_hum3'))

    # merged_df = merged_df + assw
    merged_df = pd.merge(merged_df, df_assw, on=common_cols, how='outer', suffixes=('', '_assw'))

    # check
    print("Head of the merged dataframe:")
    display(merged_df.head())

    print("Columns of the merged dataframe:")
    print(merged_df.columns.tolist())

    print(f"\nDimensions of the merged human reviews dataframe: {merged_df.shape}")

Head of the merged dataframe:


,paper_id,standardised_review_text,standardisation_status,error_message,processed_timestamp,model,temperature,reasoning_effort,chain_of_thought,standardised_review_text_hum2,...,error_message_hum2,processed_timestamp_hum2,standardised_review_text_hum3,standardisation_status_hum3,error_message_hum3,processed_timestamp_hum3,assw_text,assw_status,error_message_assw,processed_timestamp_assw
0,-0tPmzgXS5,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,human,NaN,nan,nan,Simulated Standardization: The core argument w...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
1,-0tPmzgXS5,NaN,FAILURE,Failed to process. Error: TypeError: standardi...,1.766077e+09,human,NaN,nan,nan,Simulated Standardization: The core argument w...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
2,-1x2-lp1eZf,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,human,NaN,nan,nan,Simulated Standardization: The core argument w...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
3,-1x2-lp1eZf,NaN,FAILURE,Failed to process. Error: TypeError: standardi...,1.766077e+09,human,NaN,nan,nan,Simulated Standardization: The core argument w...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
4,-4DiyBMgv9m,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,human,NaN,nan,nan,Simulated Standardization: The core argument w...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09


Columns of the merged dataframe:
['paper_id', 'standardised_review_text', 'standardisation_status', 'error_message', 'processed_timestamp', 'model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'standardised_review_text_hum2', 'standardisation_status_hum2', 'error_message_hum2', 'processed_timestamp_hum2', 'standardised_review_text_hum3', 'standardisation_status_hum3', 'error_message_hum3', 'processed_timestamp_hum3', 'assw_text', 'assw_status', 'error_message_assw', 'processed_timestamp_assw']

Dimensions of the merged human reviews dataframe: (2806, 21)


In [5]:
# @title Merge with paper metadata

# access dataframes
df_paper = dataframes.get('df_paper')

# check if dataframes exist before merging
if 'merged_df' not in locals() or merged_df is None:
    print("Error: 'merged_df' is not available. Please ensure the previous merge operation was successful.")
elif df_paper is None:
    print("Error: 'df_paper' is not loaded. Please ensure the data loading step was successful.")
else:
    # merge merged_df and df_paper on 'paper_id' with a full outer join
    merged_df = pd.merge(df_paper, merged_df, on='paper_id', how='outer')

    # check
    print("Head of the final merged dataframe (human reviews + paper metadata):")
    display(merged_df.head())

    print("Columns of the final merged dataframe:")
    print(merged_df.columns.tolist())

    print(f"\nDimensions of the final merged dataframe: {merged_df.shape}")

Head of the final merged dataframe (human reviews + paper metadata):


,paper_id,pdf_url,abstract,parsed_text,human_review1,human_review2,human_review3,standardised_review_text,standardisation_status,error_message,...,error_message_hum2,processed_timestamp_hum2,standardised_review_text_hum3,standardisation_status_hum3,error_message_hum3,processed_timestamp_hum3,assw_text,assw_status,error_message_assw,processed_timestamp_assw
0,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
1,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,NaN,FAILURE,Failed to process. Error: TypeError: standardi...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
2,-1x2-lp1eZf,https://openreview.net/pdf/7a91a6c886613eb16a1...,"By adopting deep convolution architectures, sp...",INTRODUCTION Spiking neural networks (SNNs) (M...,Summary Of The Paper:\n\nThe paper presents an...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper introduces...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
3,-1x2-lp1eZf,https://openreview.net/pdf/7a91a6c886613eb16a1...,"By adopting deep convolution architectures, sp...",INTRODUCTION Spiking neural networks (SNNs) (M...,Summary Of The Paper:\n\nThe paper presents an...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper introduces...,NaN,FAILURE,Failed to process. Error: TypeError: standardi...,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09
4,-4DiyBMgv9m,https://openreview.net/pdf/922f8c68ece0fa54603...,This paper considers the permuted linear regre...,INTRODUCTION This paper considers the permuted...,Summary Of The Paper:\n\nThis work explores th...,Summary Of The Paper:\n\nThe authors study pha...,Summary Of The Paper:\n\nThis paper considers ...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,NaN,1.765892e+09,Simulated Standardization: The core argument w...,SIMULATED,NaN,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09


Columns of the final merged dataframe:
['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3', 'standardised_review_text', 'standardisation_status', 'error_message', 'processed_timestamp', 'model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'standardised_review_text_hum2', 'standardisation_status_hum2', 'error_message_hum2', 'processed_timestamp_hum2', 'standardised_review_text_hum3', 'standardisation_status_hum3', 'error_message_hum3', 'processed_timestamp_hum3', 'assw_text', 'assw_status', 'error_message_assw', 'processed_timestamp_assw']

Dimensions of the final merged dataframe: (2806, 27)


In [6]:
# @title Merge with llm reviews

# access dataframes
df_llm = dataframes.get('df_llm')

# check if dataframes exist before merging
if 'merged_df' not in locals() or merged_df is None:
    print("Error: 'merged_df' is not available. Please ensure the previous merge operation was successful.")
elif df_llm is None:
    print("Error: 'df_llm' is not loaded. Please ensure the data loading step was successful.")
else:
    # merge merged_df on 'paper_id' and df_llm on 'document_id' with a left join
    merged_df = pd.merge(merged_df, df_llm,
                                left_on='paper_id',
                                right_on='document_id',
                                how='left',
                                suffixes=('', '_llm')) # suffixes for conflicting column names

    # drop redundant column
    if 'document_id' in merged_df.columns:
        merged_df = merged_df.drop(columns=['document_id'])

    # check
    print("Head of the final merged dataframe (human reviews + paper metadata + LLM reviews):\n(Note: This merge now joins on paper_id/document_id and repeats paper info for each LLM config.)")
    display(merged_df.head())

    print("Columns of the final merged dataframe:")
    print(merged_df.columns.tolist())

    print(f"\nDimensions of the final merged dataframe: {merged_df.shape}")

Head of the final merged dataframe (human reviews + paper metadata + LLM reviews):
(Note: This merge now joins on paper_id/document_id and repeats paper info for each LLM config.)


,paper_id,pdf_url,abstract,parsed_text,human_review1,human_review2,human_review3,standardised_review_text,standardisation_status,error_message,...,processed_timestamp_hum3,assw_text,assw_status,error_message_assw,processed_timestamp_assw,review_text,model_llm,temperature_llm,reasoning_effort_llm,chain_of_thought_llm
0,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",gpt-5-mini-2025-08-07,NaN,low,nan
1,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",gpt-5-mini-2025-08-07,NaN,low,explain your thought process step by step
2,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",gpt-5-mini-2025-08-07,NaN,high,nan
3,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",gpt-5-mini-2025-08-07,NaN,high,explain your thought process step by step
4,-0tPmzgXS5,https://openreview.net/pdf/d693e0ef8d0dfd32267...,Video recognition methods based on 2D networks...,INTRODUCTION Video recognition methods has evo...,Summary Of The Paper:\n\nThe paper proposes an...,Summary Of The Paper:\n\nThis paper makes two ...,Summary Of The Paper:\n\nThe authors propose a...,Simulated Standardization: The core argument w...,SIMULATED,NaN,...,1.765892e+09,Simulated ASSW extraction: The core argument w...,SIMULATED,NaN,1.765903e+09,"{\n ""model"": ""gpt-5-nano-2025-08-07"",\n ""tem...",gpt-5-nano-2025-08-07,NaN,low,nan


Columns of the final merged dataframe:
['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3', 'standardised_review_text', 'standardisation_status', 'error_message', 'processed_timestamp', 'model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'standardised_review_text_hum2', 'standardisation_status_hum2', 'error_message_hum2', 'processed_timestamp_hum2', 'standardised_review_text_hum3', 'standardisation_status_hum3', 'error_message_hum3', 'processed_timestamp_hum3', 'assw_text', 'assw_status', 'error_message_assw', 'processed_timestamp_assw', 'review_text', 'model_llm', 'temperature_llm', 'reasoning_effort_llm', 'chain_of_thought_llm']

Dimensions of the final merged dataframe: (157136, 32)


In [7]:
# @title Define simulation

# define LLMClient
class LLMClient:
    """
    Handles Llama-3.1 log-probability calculations using Hugging Face transformers.
    """

    # define initialisation setting
    def __init__(self, simulate: bool = True, model_id: str = MODEL_ID, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.model_id = model_id
        self.model = None
        self.tokenizer = None

        # initialize client if not simulating
        self._maybe_init_local_model()

    # define client initalisation
    def _maybe_init_local_model(self):
        """Initializes the Llama model with 4-bit quantization if not simulating."""

        # is simulating
        if self.simulate:
            print("INFO: LLMClient initialized in SIMULATION mode.")
            return

        # else load the actual model
        try:
            print(f"INFO: Loading {self.model_id} in 4-bit...")
            # load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            self.tokenizer.pad_token = self.tokenizer.eos_token

            # load model itself
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id,
                quantization_config=BNB_CONFIG, # apply 4-bit quantization
                device_map="auto", # settings
                trust_remote_code=True # settings
            )
            print("INFO: Llama model loaded successfully.")

        # else print error message
        except Exception as e:
            print(f"ERROR: Could not load local model: {e}")
            self.simulate = True

    # define log probability call
    def get_log_prob(
        self,
        user_prompt: str, # adjusted per review
        target_output: str, # adjusted per review
        temperature: int = TEMPERATURE_DEFAULT, # hardcoded as in Xu et al.
        model: str = MODEL_ID, # hardcoded as in Xu et al.
        system_prompt: str = SYSTEM_PROMPT_GEM, # hardcoded as in Xu et al.
        max_tokens: int = MAX_TOKENS, # hardcoded as in Xu et al.
        max_retries: int = MAX_RETRIES, # hardcoded
        retry_delay: float = RETRY_DELAY, # hardcoded
        ) -> Tuple[float, str]:
        """
        Calculates the log-probability of 'target_output' given the prompts.
        """

        # simulation path
        if self.simulate:
            # simulate a random log-probability (usually a negative float)
            return self.rng.uniform(-2.0, -0.1), "SIMULATED"

        # real log-probability processing
        for attempt in range(1, max_retries + 1):
            try:
                # formates and tokenizes the promtps for the model
                prompt_ids = self.tokenizer.apply_chat_template(
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    add_generation_prompt=True, # settings
                    return_tensors="pt").to(self.model.device) # settings

                # tokenize the forced output
                target_ids = self.tokenizer(
                    target_output,
                    return_tensors="pt",
                    add_special_tokens=False
                ).input_ids.to(self.model.device)

                # combine prompts and forced output for a single call
                full_input_ids = torch.cat([prompt_ids, target_ids], dim=-1)

                with torch.no_grad():
                    outputs = self.model(full_input_ids)
                    logits = outputs.logits

                # extract forced output token probabilities
                shift_logits = logits[0, prompt_ids.shape[-1]-1 : -1, :]
                shift_labels = target_ids[0, :]

                # calculate the log-probability
                log_probs = F.log_softmax(shift_logits, dim=-1)
                target_log_probs = log_probs[range(len(shift_labels)), shift_labels]

                # calculate total log-probability of the whole sequence
                total_log_prob = target_log_probs.sum().item()

                # return result
                return float(total_log_prob), "SUCCESS"

            except Exception as e:
                # error messsage in case of failure
                error_msg = f"Error (Attempt {attempt}/{max_retries}): {type(e).__name__}: {e}"

                # if maximum attempts reached
                if attempt == max_retries:
                    # error message in case of total failure
                    print(f"FATAL: {error_msg}")

                    # return placeholder value
                    return 0.0, f"FAILURE: {str(e)}"

                # if maximum attempts not reached yet
                print(f"WARNING: {error_msg}. Retrying in {retry_delay}s...")

                # delay for next attempt
                time.sleep(retry_delay)

# define helper function to construct the user prompt for GEM score calculation
def _construct_user_prompt_for_gem(
    row: pd.Series,
    include_abstract: bool, # parameter to control inclusion of the abstract
    include_first_review: bool, # parameter to control inclusion of review X
    include_assw: bool, # parameter to control inclusion of ASSW text
    prompt_review_x_col_name: Optional[str] # Column name for review X in the prompt
) -> str:
    """
    Constructs the user prompt for GEM score calculation.
    """

    user_prompt_parts = []

    # abstract
    user_prompt_parts.append("[Abstract of the paper]")
    if include_abstract and pd.notna(row['abstract']):
        user_prompt_parts.append(str(row['abstract']))
    else:
        user_prompt_parts.append("Not Available")

    # assw
    user_prompt_parts.append("[Author stated strenght and weaknesses)]")
    if include_assw and pd.notna(row['assw_text']):
        user_prompt_parts.append(str(row['assw_text']))
    else:
        user_prompt_parts.append("Not Available")

    # review X (from the first reviewer)
    user_prompt_parts.append("[Review judgments from the first reviewer]")
    if include_first_review and prompt_review_x_col_name and pd.notna(row[prompt_review_x_col_name]):
        user_prompt_parts.append(str(row[prompt_review_x_col_name]))
    else:
        user_prompt_parts.append("Not Available")

    return "\n\n".join(user_prompt_parts)


# define review standardisation function
def get_results(
    client: LLMClient,
    user_prompt_gem: str, # adjusted per review
    id_variable: str,
    target_output: str, # adjusted per review
    original_review_model: Optional[str],
    original_review_temperature: Optional[float],
    original_reasoning_effort: Optional[str] = None,
    original_chain_of_thought: Optional[str] = None,
    system_prompt: str = SYSTEM_PROMPT_GEM, # renamed for clarity and usage
) -> List[Dict[str, Any]]:
    """
    Applies the LLMClient to calculate the log-probability of the target output given the user prompt
    and binds this result with the original review metadata.
    """

    # call client as defined
    log_prob, status = client.get_log_prob(
        user_prompt=user_prompt_gem, # as defined just before
        model=MODEL_ID, # hardcoded as in Xu et al.
        system_prompt=system_prompt, # Use the parameter from this function
        target_output=target_output, # as defined just before
    )

    # initialize list
    records = []

    # append results to the list, including the original review's metadata
    records.append({
        'paper_id': id_variable,
        'forced_output': target_output, # as defined
        'log_prob': log_prob, # as returned
        'perplexity': np.exp(-log_prob) if status == "SUCCESS" else None,
        'status': status,
        'processed_timestamp': time.time(), # timestamp of processing
        'model': original_review_model, # original metadata
        'temperature': original_review_temperature, # original metadata
        'reasoning_effort': original_reasoning_effort, # original metadata
        'chain_of_thought': original_chain_of_thought # original metadata
    })

    return records

# define helper function to run get_results() on different data
def process_results(
    df: pd.DataFrame,
    config_model: Optional[str],
    config_temperature: Optional[float],
    config_reasoning_effort: Optional[str],
    config_chain_of_thought: Optional[str],
    results_path: str,
    log_path: str,
    include_abstract: bool, # new parameter for _construct_user_prompt_for_gem()
    include_first_review: bool, # new parameter for _construct_user_prompt_for_gem()
    include_assw: bool, # New parameter for _construct_user_prompt_for_gem()
    prompt_review_x_col_name: Optional[str], # Column for Review X (in the prompt)
    forced_output_review_y_col_name: str, # Column for Review Y (forced output)
    adapted_res_columns: List[str] # for dynamic result columns
):
    """
    Applies the _construct_user_prompt_for_gem()-function to create the user prompt and then applies the get_results()
    function to obtain and store the results including the metadata about applied llm-based review Y. The function returns
    also a log-file for the success across metadata configurations
    """

    # information message
    print(f"\n--- Probabilistic Analysis (Forced Output Review: '{forced_output_review_y_col_name}') ---")

    # initialize count for later messages
    papers_processed_count = 0
    papers_succeeded_count = 0

    # iterate over each paper in the provided dataframe/group
    for i, row in df.iterrows():

        # increase count
        papers_processed_count += 1

        # construct user prompt (X)
        user_prompt_gem = _construct_user_prompt_for_gem(
            row=row,
            include_abstract=include_abstract,
            include_first_review=include_first_review,
            include_assw=include_assw,
            prompt_review_x_col_name=prompt_review_x_col_name
        )

        # extract the target output (Y)
        target_output_text = str(row[forced_output_review_y_col_name])

        try:
            # apply get_results
            results = get_results(
                client=client,
                user_prompt_gem=user_prompt_gem, # as defined just before
                id_variable=row['paper_id'], # defined by dataframe
                target_output=target_output_text, # as defined just before
                original_review_model=config_model, # for logging
                original_review_temperature=config_temperature, # for logging
                original_reasoning_effort=config_reasoning_effort, # for logging
                original_chain_of_thought=config_chain_of_thought # for logging
            )

            # convert to dataframe
            df_result = pd.DataFrame(results)

            # adjust columns as hardcoded
            df_result = df_result[adapted_res_columns] # use adapted columns

            # append to result file
            df_result.to_csv(results_path, mode="a", header=False, index=False)

            # succcess message
            print(f"[SUCCESS] Processed (X: {prompt_review_x_col_name}, Y: {forced_output_review_y_col_name}) for ID {row['paper_id']} with original config (Model: {config_model}, Temp: {config_temperature}).")

            # increase count
            papers_succeeded_count += 1

        except Exception as e:
            # create artificial error record
            failure_record = {
                'paper_id': row['paper_id'],
                'forced_output': target_output_text, # actual target output
                'log_prob': None,
                'perplexity': None,
                'status': "FAILURE",
                'processed_timestamp': time.time(),
                'model': config_model, # for logging
                'temperature': config_temperature, # for logging
                'reasoning_effort': config_reasoning_effort, # for logging
                'chain_of_thought': config_chain_of_thought # for logging
            }

            # convert to dataframe
            df_result = pd.DataFrame([failure_record])

            # adjust columns as hardcoded
            df_result = df_result[adapted_res_columns] # use adapted columns

            # append to result file anyway
            df_result.to_csv(results_path, mode="a", header=False, index=False)

            # failure message
            print(f"[FAILURE] Failed to process (X: {prompt_review_x_col_name}, Y: {forced_output_review_y_col_name}) for ID {row['paper_id']} with original config (Model: {config_model}, Temp: {config_temperature}). Error: {e}")

    # create log directory
    log_entry = {
        'model': config_model, # as in df_result
        'temperature': config_temperature, # as in df_result
        'reasoning_effort': config_reasoning_effort, # as in df_result
        'chain_of_thought': config_chain_of_thought, # as in df_result
        'papers_processed_count': papers_processed_count, # as result of iterative count
        'papers_succeeded_count': papers_succeeded_count, # as result of iterative count
        'configuration_status': "SUCCESS" if papers_succeeded_count == papers_processed_count else "PARTIAL_FAILURE" if papers_succeeded_count > 0 else "TOTAL_FAILURE"
    }

    # convert to dataframe
    log_df = pd.DataFrame([log_entry])

    # append to log file
    log_df[BASE_LOG_COLUMNS].to_csv(log_path, mode="a", header=False, index=False)

    # information message
    print(f"Logged status for (X: {prompt_review_x_col_name}, Y: {forced_output_review_y_col_name}) (Model: {config_model}, Temp: {config_temperature}): {log_entry['configuration_status']} (Processed {papers_succeeded_count}/{papers_processed_count} papers).")

In [8]:
# @title Helper function for flexible storage

def construct_gem_filepaths(include_abstract: bool, include_first_review: bool, include_assw: bool, result_dir: str, base_res_columns: list):
    """
    Constructs the GEM score result and log file paths and adapts column names based on inclusion flags.

    Args:
        include_abstract (bool): Whether the abstract is included in the prompt (for '_s' suffix).
        include_first_review (bool): Whether the first review is included in the prompt (for '_con' or '_mar' suffix).
        include_assw (bool): Whether the ASSW text is included in the prompt (for '_assw' suffix).
        raw_dir (str): The base directory for raw data.
        base_res_columns (list): The base list of result column names to adapt.

    Returns:
        tuple[str, str, list[str]]: A tuple containing (results_path, log_path, adapted_result_columns).
    """

    # creates suffixes
    abstract_suffix = "_s" if include_abstract else ""
    review_suffix = "_con" if include_first_review else "_mar"
    assw_suffix = "_assw" if include_assw else ""

    # constructs filenames
    results_filename = f"gem{abstract_suffix}{assw_suffix}{review_suffix}_res.csv"
    log_filename = f"gem{abstract_suffix}{assw_suffix}{review_suffix}_log.csv"

    # creates paths
    result_path = os.path.join(result_dir, results_filename)
    log_path = os.path.join(result_dir, log_filename)

    # adapt result column
    adapted_result_columns = list(base_res_columns)
    try:
        log_prob_index = adapted_result_columns.index('log_prob')
        suffix_for_log_prob = f"{abstract_suffix}{assw_suffix}{review_suffix}"
        adapted_result_columns[log_prob_index] = f"log_prob{suffix_for_log_prob}"
    except ValueError:
        print("Warning: 'log_prob' not found in BASE_RES_COLUMNS. No suffix applied.")

    return result_path, log_path, adapted_result_columns

In [ ]:
# @title Probability calculation

# initialize the client
client = LLMClient(simulate=SIMULATE)

# Iterate over each predefined calculation setting
for setting_name, config in CALCULATION_SETTINGS.items():
    print(f"\n--- Starting Calculation for Setting: {setting_name} ---")

    include_abstract_flag = config["include_abstract_flag"]
    include_assw_flag = config["include_assw_flag"]
    include_first_review_flag = config["include_first_review_flag"]

    # Construct paths and adapted columns for the current setting
    RESULT_PATH_CURRENT, LOG_PATH_CURRENT, ADAPTED_RES_COLUMNS_CURRENT = construct_gem_filepaths(
        include_abstract=include_abstract_flag,
        include_first_review=include_first_review_flag,
        include_assw=include_assw_flag,
        result_dir=RESULT_DIR,
        base_res_columns=BASE_RES_COLUMNS
    )

    # create new result file (if not existing) with appropriate headers
    if not os.path.exists(RESULT_PATH_CURRENT):
        with open(RESULT_PATH_CURRENT, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(ADAPTED_RES_COLUMNS_CURRENT)
        print(f"Created result file: {RESULT_PATH_CURRENT}")

    # create new log file (if not existing) with appropriate headers
    if not os.path.exists(LOG_PATH_CURRENT):
        with open(LOG_PATH_CURRENT, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(BASE_LOG_COLUMNS)
        print(f"Created log file: {LOG_PATH_CURRENT}")

    # Group the merged_df by the LLM configuration columns
    grouped_merged_df = merged_df.groupby(LLM_CONFIG_COLS)

    # Iterate over each group (unique LLM configuration)
    for config_key, group_df in grouped_merged_df:
        # Extract configuration parameters from the group key
        config_model, config_temperature, config_reasoning_effort, config_chain_of_thought = config_key

        # Assign LLM review for the prompt (Review X)
        # This will be 'review_text' from the merged_df
        current_prompt_review_x_col_name = TARGET_COL # 'review_text' (LLM output)

        # Assign a randomly selected human review for the forced output (Review Y)
        current_forced_output_review_y_col_name = random.choice(HUMAN_REVIEW_COLS)

        # Call process_results for this entire group of papers under the current configuration
        process_results(
            df=group_df, # Pass the DataFrame for the current configuration group
            config_model=config_model,
            config_temperature=config_temperature,
            config_reasoning_effort=config_reasoning_effort,
            config_chain_of_thought=config_chain_of_thought,
            results_path=RESULT_PATH_CURRENT,
            log_path=LOG_PATH_CURRENT,
            include_abstract=include_abstract_flag,
            include_first_review=include_first_review_flag,
            include_assw=include_assw_flag, # Pass the new flag
            prompt_review_x_col_name=current_prompt_review_x_col_name, # LLM review as X
            forced_output_review_y_col_name=current_forced_output_review_y_col_name, # Human review as Y
            adapted_res_columns=ADAPTED_RES_COLUMNS_CURRENT # Pass the adapted columns
        )
    print(f"--- Finished Calculation for Setting: {setting_name} ---")